# 0. verify the file

In [ ]:
import os
file_path = "/content/gameplay.mp4"
if os.path.exists(file_path):
    size_kb = os.path.getsize(file_path) / 1024
    print(f"File size is: {size_kb:.2f} KB")
    if size_kb < 1000:
        print("This file is too small to be a video. The download failed.")

# 1. ENVIRONMENT SETUP & DEPENDENCY INSTALLATION

In [ ]:
!pip install -q openai-whisper librosa moviepy opencv-python-headless numpy scipy tqdm

import os
import sys
import torch
import cv2
import librosa
import whisper
import numpy as np
from scipy.io import wavfile
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_videoclips, CompositeAudioClip
from tqdm import tqdm

# Verify hardware acceleration to maximize local execution speed
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")

# 2. COLAB FORM CONFIGURATION & USER INPUTS


In [ ]:
VIDEO_INPUT_PATH = "/content/gameplay.mp4"
BGM_INPUT_PATH = "/content/music.mp3"
OUTPUT_DIR = "/content/output"
FINAL_VIDEO_NAME = "ai_montage.mp4"

# Pipeline Tuning Parameters
TARGET_MONTAGE_DURATION = 180 # Adjust up to 300.
CLIP_PADDING_SECONDS = 2.0
VISUAL_MOTION_THRESHOLD = 0.4
AUDIO_PEAK_THRESHOLD = 0.5

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Pipeline configured for a {TARGET_MONTAGE_DURATION}-second montage.")
print(f"Target Video: {VIDEO_INPUT_PATH}")
print(f"Target Audio: {BGM_INPUT_PATH}")

# 3. DIRECTORY ARCHITECTURE & INITIALIZATION

In [ ]:
import shutil

TEMP_DIR = os.path.join(OUTPUT_DIR, "pipeline_temp")
CLIPS_DIR = os.path.join(TEMP_DIR, "extracted_clips")

# Reset workspace directories to avoid caching conflicts from previous runs
for folder in [TEMP_DIR, CLIPS_DIR]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

EXTRACTED_VOICE_PATH = os.path.join(TEMP_DIR, "extracted_voice.wav")
FINAL_OUTPUT_PATH = os.path.join(OUTPUT_DIR, FINAL_VIDEO_NAME)

print(f"Workspace clean. Temporary assets isolated in: {TEMP_DIR}")

# 4: AUDIO EXTRACTION ENGINE


In [ ]:
import os
from moviepy.editor import VideoFileClip

def extract_native_audio(video_path, output_audio_path):
    print(f"Extracting native audio layer from: {video_path}")
    try:
        video = VideoFileClip(video_path)
        if video.audio is not None:
            # Export as a standardized 16kHz mono WAV file for optimal ML processing
            video.audio.write_audiofile(
                output_audio_path,
                fps=16000,
                nbytes=2,
                buffersize=2000,
                codec="pcm_s16le",
                ffmpeg_params=["-ac", "1"]
            )
            video.close()
            print("Audio layer successfully isolated.")
            return True
        else:
            video.close()
            print("Error: Input video track contains no audio channel.")
            return False
    except Exception as e:
        print(f"Audio extraction failure: {str(e)}")
        return False

# Defensive check before proceeding down-funnel
if os.path.exists(VIDEO_INPUT_PATH):
    extraction_success = extract_native_audio(VIDEO_INPUT_PATH, EXTRACTED_VOICE_PATH)
else:
    print(f"Execution Halted: {VIDEO_INPUT_PATH} not found. Please ensure the file is uploaded.")
    extraction_success = False

# 5: AUDIO NORMALIZATION & PREPROCESSING


In [ ]:
def load_and_normalize_audio(audio_path):
    print("Loading audio for analytical processing...")
    # Load via Librosa to ensure uniform sampling and float conversion
    y, sr = librosa.load(audio_path, sr=16000)

    # Apply root-mean-square peak normalization to mitigate systemic recording variances
    max_peak = np.max(np.abs(y))
    if max_peak > 0:
        y_normalized = y / max_peak
    else:
        y_normalized = y

    print(f"Audio normalized across {len(y_normalized)} samples at {sr}Hz.")
    return y_normalized, sr

if extraction_success and os.path.exists(EXTRACTED_VOICE_PATH):
    norm_audio, sampling_rate = load_and_normalize_audio(EXTRACTED_VOICE_PATH)
else:
    norm_audio, sampling_rate = None, 16000

# 6: DYNAMIC AUDIO PEAK DETECTION (for no comentry videos)

In [ ]:
import librosa
import numpy as np
import os

def detect_action_spikes(audio_path):
    print("No Commentary Mode: Scanning audio track for dynamic volume peaks (gunshots)...")

    y, sr = librosa.load(audio_path, sr=22050)
    hop_length = 512
    energy = np.array([sum(abs(y[i:i+hop_length]**2)) for i in range(0, len(y), hop_length)])

    if np.max(energy) == 0:
        return []

    energy = energy / np.max(energy)

    # Automatically find the top 15% loudest moments for this specific video
    dynamic_threshold = np.percentile(energy[energy > 0], 85)
    print(f"Dynamic volume threshold set to: {dynamic_threshold:.3f}")

    peak_frames = np.where(energy > dynamic_threshold)[0]

    action_events = []
    times = librosa.frames_to_time(peak_frames, sr=sr, hop_length=hop_length)

    current_start, current_end = None, None

    for t in times:
        if current_start is None:
            current_start, current_end = t, t
        elif t - current_end < 1.0:
            current_end = t
        else:
            action_events.append({"start": current_start, "end": current_end, "score": 1.5, "type": "gunshot_peak"})
            current_start, current_end = t, t

    if current_start is not None:
         action_events.append({"start": current_start, "end": current_end, "score": 1.5, "type": "gunshot_peak"})

    print(f"Dynamic detection isolated {len(action_events)} major audio spikes.")

    # Keeping the variable name 'verbal_events' so the rest of the pipeline connects perfectly
    return action_events

if extraction_success and os.path.exists(EXTRACTED_VOICE_PATH):
    verbal_events = detect_action_spikes(EXTRACTED_VOICE_PATH)
else:
    verbal_events = []

# 7: VISUAL MOTION DETECTION (with CPU)


In [ ]:
import cv2
from tqdm import tqdm
import numpy as np
import os

def analyze_visual_motion(video_path, sample_fps=2):
    print("Initiating robust OpenCV spatial-temporal mapping...")

    # Force FFmpeg backend for better compatibility in Colab
    cap = cv2.VideoCapture(video_path, cv2.CAP_FFMPEG)

    if not cap.isOpened():
        print("CRITICAL WARNING: OpenCV could not open the video file.")
        print("Fallback activated: Ignoring visual data and relying entirely on audio spikes.")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    if total_frames == 0 or video_fps == 0:
        print("CRITICAL WARNING: Video file has 0 frames or 0 FPS (likely corrupted).")
        cap.release()
        return []

    frame_step = max(1, int(video_fps / sample_fps))
    motion_scores = []

    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        return []

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    frame_index = 0

    # Read a maximum of 5000 frames to prevent Colab RAM crashes on long videos
    max_frames_to_read = min(total_frames, 5000 * frame_step)

    with tqdm(total=max_frames_to_read, desc="Processing Frames") as pbar:
        while frame_index < max_frames_to_read:
            ret, frame = cap.read()
            if not ret:
                break

            frame_index += 1
            if frame_index % frame_step != 0:
                continue

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame_delta = cv2.absdiff(prev_gray, gray)
            _, thresh = cv2.threshold(frame_delta, 25, 255, cv2.THRESH_BINARY)

            motion_ratio = np.sum(thresh == 255) / float(thresh.size)
            motion_scores.append({"time": frame_index / video_fps, "score": motion_ratio})

            prev_gray = gray
            pbar.update(frame_step)

    cap.release()
    print(f"Visual processing complete. Profiled {len(motion_scores)} spatial nodes.")
    return motion_scores

if os.path.exists(VIDEO_INPUT_PATH):
    visual_motion_profile = analyze_visual_motion(VIDEO_INPUT_PATH)
else:
    visual_motion_profile = []

# 7: VISUAL MOTION DETECTION (GPU)

In [ ]:
import cv2
import torch
from tqdm import tqdm
import os

def analyze_visual_motion_gpu(video_path, sample_fps=2):
    print("Initiating GPU-Accelerated Spatial Mapping...")

    # Force the pipeline to target the T4 GPU VRAM
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Compute Engine Active: {device.type.upper()}")

    cap = cv2.VideoCapture(video_path, cv2.CAP_FFMPEG)

    if not cap.isOpened():
        print("CRITICAL WARNING: Could not open the video file.")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    if total_frames == 0 or video_fps == 0:
        cap.release()
        return []

    frame_step = max(1, int(video_fps / sample_fps))
    motion_scores = []

    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        return []

    # Convert the first frame to grayscale and push it directly into the GPU VRAM
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    prev_tensor = torch.tensor(prev_gray, device=device, dtype=torch.float32)

    frame_index = 0

    with tqdm(total=total_frames, desc="GPU Processing Frames") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_index += 1
            if frame_index % frame_step != 0:
                continue

            # Convert frame, push to GPU
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            curr_tensor = torch.tensor(gray, device=device, dtype=torch.float32)

            # Perform the heavy mathematical differencing inside the T4 VRAM
            frame_delta = torch.abs(prev_tensor - curr_tensor)

            # Apply threshold and calculate motion ratio natively on the GPU
            thresh = (frame_delta > 25.0).float()
            motion_ratio = torch.mean(thresh).item()

            motion_scores.append({"time": frame_index / video_fps, "score": motion_ratio})

            # Cycle the tensors to free up VRAM continuously
            prev_tensor = curr_tensor
            pbar.update(frame_step)

    cap.release()

    # Flush the VRAM cache so the rest of the pipeline has room to operate
    torch.cuda.empty_cache()

    print(f"GPU Visual processing complete. Profiled {len(motion_scores)} spatial nodes.")
    return motion_scores

if os.path.exists(VIDEO_INPUT_PATH):
    visual_motion_profile = analyze_visual_motion_gpu(VIDEO_INPUT_PATH)
else:
    visual_motion_profile = []

# 8: AUDIO BEAT & RHYTHM TRACKING (LIBROSA)


In [ ]:
def detect_musical_cadence(bgm_path):
    print(f"Analyzing background musical infrastructure: {bgm_path}")
    if not os.path.exists(bgm_path):
        print("Warning: Background music file absent. Generating rhythmic grid manually.")
        return np.arange(0, TARGET_MONTAGE_DURATION, 0.5)

    y, sr = librosa.load(bgm_path, sr=22050)
    # Isolate global tempo and corresponding frame metrics
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr)

    # FIX: Handle newer Librosa versions that return tempo as a numpy array
    import numpy as np
    tempo_value = float(tempo[0]) if isinstance(tempo, np.ndarray) else float(tempo)

    print(f"Rhythmic analysis complete. Detected Tempo: {tempo_value:.2f} BPM. Mapped {len(beat_times)} active beat nodes.")
    return beat_times

musical_beats = detect_musical_cadence(BGM_INPUT_PATH)

# 9: DISTINCT ACTION-CLIP ISOLATION

In [ ]:
def fuse_analysis_streams(video_path, action_events, motion_profile, padding, max_duration):
    from moviepy.editor import VideoFileClip
    print("Initiating distinct action-to-clip mapping...")

    try:
        clip = VideoFileClip(video_path)
        video_duration = clip.duration
        clip.close()
    except Exception:
        video_duration = 10000

    # Sort the 432 action spikes to grab the absolute loudest ones first
    action_events.sort(key=lambda x: x["score"], reverse=True)

    selected_segments = []
    allocated_time = 0

    # We will force a rigid 3-second window for every gunshot to ensure fast pacing
    strict_padding = 1.5

    for event in action_events:
        if allocated_time >= max_duration:
            break

        center = (event["start"] + event["end"]) / 2.0
        start = max(0, center - strict_padding)
        end = min(video_duration, center + strict_padding)

        # Prevent clips from merging. If a shot happens inside an existing clip, skip it.
        overlap = False
        for (s, e) in selected_segments:
            if start < e and end > s:
                overlap = True
                break

        if not overlap:
            selected_segments.append((start, end))
            allocated_time += (end - start)

    # Sort back into chronological order so the gameplay progresses naturally
    selected_segments.sort(key=lambda x: x[0])

    # Format for Phase 10
    final_segments = [(s[0], s[1], 1.0) for s in selected_segments]

    print(f"Successfully isolated {len(final_segments)} distinct action clips.")
    return final_segments

if os.path.exists(VIDEO_INPUT_PATH):
    highlight_segments = fuse_analysis_streams(
        VIDEO_INPUT_PATH, verbal_events, visual_motion_profile, CLIP_PADDING_SECONDS, TARGET_MONTAGE_DURATION
    )
else:
    highlight_segments = []

# 10: SMART TRIM & SUB-CLIP GENERATION (MOVIEPY)


In [ ]:
def execute_precision_trimming(video_path, segments, target_dir):
    print("Initiating sub-clip multi-threaded extraction layout...")
    extracted_paths = []
    video = VideoFileClip(video_path)

    for idx, (start, end, _) in enumerate(segments):
        clip_name = f"highlight_{idx:03d}.mp4"
        out_path = os.path.join(target_dir, clip_name)

        print(f"Slicing Segment {idx}: Runtime [{start:.2f}s -> {end:.2f}s]")
        sub_clip = video.subclip(start, end)
        # Force write with identical structural containers
        sub_clip.write_videofile(
            out_path,
            codec="libx264",
            audio_codec="aac",
            logger=None,
            ffmpeg_params=["-crf", "23", "-preset", "fast"]
        )
        extracted_paths.append(out_path)

    video.close()
    print(f"Extraction step successful. Generated {len(extracted_paths)} clean sub-clips.")
    return extracted_paths

if highlight_segments:
    clip_manifest = execute_precision_trimming(VIDEO_INPUT_PATH, highlight_segments, CLIPS_DIR)
else:
    clip_manifest = []

# 11: BEAT-SYNCED VIDEO STITCHING ENGINE (system ram > 16GB)
The System RAM Depends on the video length, longer the video higher the ram usage

In [ ]:
def synchronize_clips_to_cadence(clip_paths, beat_nodes):
    print("Aligning video transitions to musical beat nodes via hard-trimming...")
    stitched_clips = []
    current_timeline = 0.0

    import numpy as np

    for path in clip_paths:
        raw_clip = VideoFileClip(path)
        clip_dur = raw_clip.duration

        # Find the next logical beat in the track
        valid_beats = beat_nodes[beat_nodes > current_timeline + 2.0]

        if len(valid_beats) > 0:
            next_beat = valid_beats[0]
            adjusted_dur = next_beat - current_timeline
        else:
            adjusted_dur = clip_dur

        # THE FIX:
        # 1. Do not use fl_time (time-stretching).
        # 2. Hard-trim the clip instead.
        # 3. Apply a 0.1 second safety margin so MoviePy never hits the exact EOF.
        final_dur = min(clip_dur - 0.1, adjusted_dur)

        # Force strict limits on the video
        modified_clip = raw_clip.subclip(0, final_dur)

        # Ensure the audio track strictly obeys the new duration without stretching
        if modified_clip.audio is not None:
            safe_audio = modified_clip.audio.subclip(0, final_dur)
            safe_audio = safe_audio.set_duration(final_dur)
            modified_clip = modified_clip.set_audio(safe_audio)

        modified_clip = modified_clip.set_duration(final_dur)

        stitched_clips.append(modified_clip)
        current_timeline += final_dur

    print(f"Temporal sync executed successfully across {len(stitched_clips)} nodes.")
    return stitched_clips

if clip_manifest:
    synced_clip_objects = synchronize_clips_to_cadence(clip_manifest, musical_beats)
else:
    synced_clip_objects = []

# 12: AUDIO DUCKING & MASTER AUDIO MIXING (for ram> 16 GB)
The System RAM Depends on the video length, longer the video higher the ram usage

In [ ]:
def mix_master_audio(video_sequence, bgm_path, target_duration):
    print("Constructing multi-channel master audio mix...")

    # Consolidate raw audio elements from original source timeline
    original_audio = video_sequence.audio

    if os.path.exists(bgm_path):
        bgm_track = AudioFileClip(bgm_path).subclip(0, target_duration)
        # Apply strict attenuation matrix definitions to balance layers
        bgm_track = bgm_track.volumex(0.25) # Attenuate background track to preserve vocal layer clarity

        if original_audio is not None:
            ducked_original = original_audio.volumex(0.85)
            mixed_audio = CompositeAudioClip([ducked_original, bgm_track])
        else:
            mixed_audio = bgm_track
    else:
        mixed_audio = original_audio

    return mixed_audio

# 13: AUTO-GENERATED SUBTITLES & OVERLAY RENDERING (for ram> 16 GB)
The System RAM Depends on the video length, longer the video higher the ram usage


In [ ]:
def build_filter_complex_metadata(output_path):
    print("Colab environment detected. Bypassing FFmpeg text filters to prevent BrokenPipeError.")
    # Returning an empty array prevents FFmpeg from trying to use the missing drawtext module
    return []

# 14: QUALITY ASSURANCE & COMPRESSION (FFMPEG). (for ram> 16 GB)
The System RAM Depends on the video length, longer the video higher the ram usage

In [ ]:
def compile_and_compress_montage(clip_objects, bgm_path, final_out_path):
    print("Assembling structural timeline composition...")
    base_composition = concatenate_videoclips(clip_objects, method="compose")

    master_duration = base_composition.duration
    composite_audio = mix_master_audio(base_composition, bgm_path, master_duration)

    base_composition = base_composition.set_audio(composite_audio)

    # Grab the empty array from Phase 13
    extra_flags = build_filter_complex_metadata(final_out_path)

    # Clean render parameters without any custom video filters
    render_params = ["-crf", "20", "-preset", "fast", "-pix_fmt", "yuv420p"] + extra_flags

    print("Executing final compilation pass to Colab disk...")
    base_composition.write_videofile(
        final_out_path,
        codec="libx264",
        audio_codec="aac",
        audio_bitrate="192k",
        ffmpeg_params=render_params,
        threads=4,
        logger="bar"
    )

    base_composition.close()
    print(f"Production pipeline terminated successfully. File written to: {final_out_path}")

if synced_clip_objects:
    compile_and_compress_montage(synced_clip_objects, BGM_INPUT_PATH, FINAL_OUTPUT_PATH)
else:
    print("Compilation Halted: Highlight verification empty.")

# 15: EXECUTION TRIGGER & EXPORT PIPELINE (for ram> 16 GB)
The System RAM Depends on the video length, longer the video higher the ram usage

In [ ]:
if os.path.exists(FINAL_OUTPUT_PATH):
    file_size_mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    print("==================================================================")
    print("   PIPELINE PROCESSING EXECUTED SUCCESSFULLY")
    print("==================================================================")
    print(f"   Final Export Path: {FINAL_OUTPUT_PATH}")
    print(f"   File Size:         {file_size_mb:.2f} MB")
    print("   Ready for download or deployment via the sidebar file browser.")
    print("==================================================================")
else:
    print("Critical Failure: Final output package validation failed.")

# 11: BEAT-SYNCED MATH ENGINE (ZERO-RAM MODE)

In [ ]:
import os
from moviepy.editor import VideoFileClip

def synchronize_clips_to_cadence(clip_paths, beat_nodes):
    print("Calculating beat cuts and generating zero-RAM assembly list...")
    current_timeline = 0.0

    # We create a text file for FFmpeg instead of keeping objects in RAM
    concat_file_path = os.path.join(TEMP_DIR, "concat_list.txt")

    with open(concat_file_path, "w") as f:
        for path in clip_paths:
            # Open just to read duration, then IMMEDIATELY close to clear RAM
            temp_clip = VideoFileClip(path)
            clip_dur = temp_clip.duration
            temp_clip.close()

            valid_beats = beat_nodes[beat_nodes > current_timeline + 2.0]

            if len(valid_beats) > 0:
                next_beat = valid_beats[0]
                adjusted_dur = next_beat - current_timeline
            else:
                adjusted_dur = clip_dur

            final_dur = min(clip_dur - 0.1, adjusted_dur)

            # Write instructions directly to the text file
            f.write(f"file '{path}'\n")
            f.write(f"outpoint {final_dur:.3f}\n")

            current_timeline += final_dur

    print(f"Temporal sync math executed safely across {len(clip_paths)} nodes.")
    return concat_file_path, current_timeline

if clip_manifest:
    concat_txt_path, master_duration = synchronize_clips_to_cadence(clip_manifest, musical_beats)
else:
    concat_txt_path, master_duration = None, 0

# 12: MASTER RENDER
## after running the zero ram mode

In [ ]:
import subprocess
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

def compile_final_montage(concat_txt, bgm_path, out_path, target_duration):
    print("Step 1: Assembling raw video sequence (Bypassing RAM limits)...")
    raw_assembled = os.path.join(TEMP_DIR, "raw_assembled.mp4")

    # Use native FFmpeg to stitch the clips instantly via the text file
    cmd_concat = [
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', concat_txt,
        '-c', 'copy',
        raw_assembled
    ]
    process = subprocess.run(cmd_concat, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    if not os.path.exists(raw_assembled):
        print("CRITICAL ERROR in FFmpeg Assembly:")
        print(process.stderr.decode('utf-8'))
        return

    print("Step 2: Mixing background music and rendering final output...")
    # Safely open ONE single assembled video instead of 60 individual ones
    master_video = VideoFileClip(raw_assembled)
    original_audio = master_video.audio

    if os.path.exists(bgm_path):
        bgm_clip_full = AudioFileClip(bgm_path)
        actual_bgm_duration = bgm_clip_full.duration
        # Use min to ensure we don't go beyond the BGM's actual length
        bgm_track_duration = min(target_duration, actual_bgm_duration)
        bgm_track = bgm_clip_full.subclip(0, bgm_track_duration).volumex(0.25)
        # Removed: bgm_clip_full.close() to prevent premature closing of the reader

        audio_clips_to_mix = []
        if original_audio is not None:
            try:
                # Attempt to get a frame to verify original_audio reader functionality
                original_audio.get_frame(0)
                ducked_original = original_audio.volumex(0.85)
                audio_clips_to_mix.append(ducked_original)
            except AttributeError:
                print("Warning: Original video audio stream is unreadable by MoviePy and will be skipped.")
            except Exception as e:
                print(f"Warning: Issue with original video audio stream: {e}. Skipping original audio.")

        audio_clips_to_mix.append(bgm_track)

        if audio_clips_to_mix:
            # If only background music is present, it's not a composite clip
            if len(audio_clips_to_mix) == 1:
                mixed_audio = audio_clips_to_mix[0]
            else:
                mixed_audio = CompositeAudioClip(audio_clips_to_mix)
            master_video = master_video.set_audio(mixed_audio)
        else:
            master_video = master_video.set_audio(None) # Explicitly set no audio

    # Final render to disk
    render_params = ["-crf", "20", "-preset", "fast", "-pix_fmt", "yuv420p"]

    master_video.write_videofile(
        out_path,
        codec="libx264",
        audio_codec="aac",
        audio_bitrate="192k",
        ffmpeg_params=render_params,
        threads=4,
        logger="bar"
    )

    master_video.close()
    bgm_clip_full.close() # Close the full BGM clip only after write_videofile is complete
    print(f"Production complete. Output written to: {out_path}")

if concat_txt_path:
    compile_final_montage(concat_txt_path, BGM_INPUT_PATH, FINAL_OUTPUT_PATH, master_duration)
else:
    print("Compilation Halted: Highlight verification empty.")

# FORCE DOWNLOAD SCRIPT


In [ ]:
from google.colab import files
import os

FINAL_OUTPUT_PATH = "/content/output/ai_montage.mp4"

if os.path.exists(FINAL_OUTPUT_PATH):
    print("Initiating forced download. Please wait, your browser will prompt you to save the file shortly...")
    files.download(FINAL_OUTPUT_PATH)
else:
    print("File not found. Please ensure Phase 14 completed successfully.")

# PIPELINE DIAGNOSTIC X-RAY

In [ ]:
print("--- PIPELINE HEALTH REPORT ---")
try:
    print(f"1. Audio Spikes Detected (Phase 6): {len(verbal_events)}")
except: print("1. Phase 6 data missing.")

try:
    print(f"2. Visual Motion Nodes (Phase 7):   {len(visual_motion_profile)}")
except: print("2. Phase 7 data missing.")

try:
    print(f"3. Highlights Kept (Phase 9):       {len(highlight_segments)}")
except: print("3. Phase 9 data missing.")

try:
    print(f"4. Clips After Sync (Phase 11):     {len(synced_clip_objects)}")
    total_time = sum([c.duration for c in synced_clip_objects])
    print(f"   -> Estimated Final Length:       {total_time:.2f} seconds")
except: print("4. Phase 11 data missing.")
print("------------------------------")